RNN - Erro dos pesos computados e usado somente durante a iteração

In [ ]:
import numpy as np
from numpy import linalg as LA
import pandas as pd
import operator as op
import ipynbname
import math
import matplotlib.cm as cm
import optuna
from optuna.samplers import RandomSampler
from optuna.samplers import TPESampler
from optuna.visualization import plot_parallel_coordinate
from optuna.visualization import plot_pareto_front
from optuna.importance import get_param_importances
from optuna.exceptions import TrialPruned
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
import matplotlib as mpl
#from Testing.RTLO import *
from Functions.RLS import *
from Functions.Utils_RTLO import *
from Functions.Graphs import *
from sklearn.metrics import root_mean_squared_error as RMSE
from sklearn.metrics import mean_absolute_percentage_error as MAPE
from sklearn.metrics import mean_squared_error as MSE

FileName = ipynbname.name()

params = [14, 12, 2, 0.01, 1e-08, 0.0001, 2]
df = pd.read_csv(r'Dataset\Bearing1_1.csv')
sig = df['PC1'].values

def PlotPredError(rtlo,w=9,h=3):
    s = len(rtlo.yWAPE)
    t = rtlo.t
    fig, axes = plt.subplots(nrows=1, ncols=4, figsize=(w, h))
    axes = axes.flatten()
    ax1,ax2,ax3,ax4 = axes[0], axes[1], axes[2], axes[3]

    ax1.plot(t, rtlo.yR, color='black',label='Y-Real', linestyle='-')
    ax1.plot(t, rtlo.yP, color='blue',label='Y-Pred', linestyle='-')
    ax1.plot(t, rtlo.yL, color='blue', linestyle='--')
    ax1.plot(t, rtlo.yU, color='blue', linestyle='--')
    
    ax1.set_title('Y - Real x Prediction')
    ax1.set_xlabel('X')
    ax1.set_ylabel('Y', color='black')
    ax1.legend()

    ax2.plot(t, rtlo.eR, color='black',label='e-Real', linestyle='-')
    ax2.plot(t, rtlo.eP, color='blue',label='e-Pred', linestyle='-')
    ax2.set_title('Error - Real x Prediction')
    ax2.set_xlabel('X')
    ax2.set_ylabel('Prediction Error', color='black') 
    ax2.legend()
    
    ax3.plot(t[-s:], rtlo.yWAPE, color='blue',label='WAPE', linestyle='-')
    ax3.set_title('Prediction WAPE')
    ax3.set_xlabel('X')
    ax3.set_ylabel('WAPE', color='black') 

    '''ax4.plot(t[-s:], rtlo.rWAPE, color='blue',label='WAPE', linestyle='-')
    ax4.set_title('RUL Prediction WAPE')
    ax4.set_xlabel('X')
    ax4.set_ylabel('WAPE', color='black') '''

    fig.tight_layout()  # otherwise the right y-label is slightly clipped
    plt.show()

def PlotPredErrorPLY(rtlo, w=800, h=300):
    # s: tamanho do vetor WAPE (caso comece depois do início)
    t = rtlo.t
    
    # Criando o layout de 1 linha e 4 colunas
    fig = make_subplots(
        rows=1, cols=3, 
        shared_xaxes=True,
        subplot_titles=('Y - Real x Prediction', 'RUL - Real x Pred', 'Error - Real x Pred', 'Prediction WAPE', 'RUL Prediction WAPE')
    )

    # --- Subplot 1: Y Real x Pred (com Intervalos) ---
    fig.add_trace(go.Scatter(x=t, y=rtlo.yR, name='Y-Real', line=dict(color='black')), row=1, col=1)
    fig.add_trace(go.Scatter(x=t, y=rtlo.yP, name='Y-Pred', line=dict(color='blue')), row=1, col=1)
    # Intervalos (Dashed)
    fig.add_trace(go.Scatter(x=t, y=rtlo.yL, name='y-Lower', line=dict(color='blue', dash='dash'), showlegend=False), row=1, col=1)
    fig.add_trace(go.Scatter(x=t, y=rtlo.yU, name='y-Upper', line=dict(color='blue', dash='dash'), showlegend=False), row=1, col=1)

    fig.add_trace(go.Scatter(x=t, y=rtlo.rulR, name='rul R', line=dict(color='black'), showlegend=False), row=1, col=2)
    fig.add_trace(go.Scatter(x=t, y=rtlo.rulP, name='rul P', line=dict(color='blue'), showlegend=False), row=1, col=2)
    fig.add_trace(go.Scatter(x=t, y=rtlo.rulL, name='rul L', line=dict(color='blue'), showlegend=False), row=1, col=2)
    fig.add_trace(go.Scatter(x=t, y=rtlo.rulU, name='rul U', line=dict(color='blue'), showlegend=False), row=1, col=2)

    fig.add_trace(go.Scatter(x=t, y=rtlo.εM_hist, name='rul R', line=dict(color='black'), showlegend=False), row=1, col=3)

    # Atualizando Layout e Eixos
    fig.update_layout(
        width=w, height=h,
        title_text=f"RTLO Model Performance Analysis",
        template='plotly_white',
        showlegend=True,
        margin=dict(l=40, r=40, t=80, b=40)
    )

    # Labels dos eixos (opcional, já que os títulos ajudam)
    fig.update_xaxes(title_text="Time / Index")
    fig.update_yaxes(title_text="Amplitude", col=1)
    fig.update_yaxes(title_text="Error", col=2)

    fig.show()
    
def SelSampler(mode='auto'):
    '''mode: auto, random,  tpe'''
    if mode == 'auto':
        sampler = None
    elif mode == 'tpe':
        sampler = optuna.samplers.TPESampler(multivariate=True, constant_linker=True,group=True,n_startup_trials=2000)
    elif mode == 'random':
        sampler=RandomSampler()
    return sampler

In [48]:
class RTLO:
    def __init__(self, nI,nR,nO,ηS=[0.1,0.1,0.1], τ=10,lr=1e-5):
        np.random.seed(42)
        self.k = 1
        self.j = nI-1
        self.t = np.array([])
        self.ref = None
        self.act = 'tanh'
        self.nI, self.nR, self.nO = nI, nR, nO

        self.ηS = np.array(ηS)
        self.τ = τ
        self.ρ = 0.1

        self.xPi = np.zeros(nI)
        self.hP, self.hU, self.hL = [np.zeros(nR) for i in range(3)]

        self.pS = np.zeros((self.nR, self.nR))
        self.qS = np.zeros((self.nR, self.nI))

        self.ΔOS = np.zeros((nO, nR))
        self.ΔRS = np.zeros((nR, nR))
        self.ΔIS = np.zeros((nR, nI))
        
        self.wI = XavierUniform([nR, nI],sd=42)
        self.wR = XavierUniform([nR, nR],sd=41)
        self.wO = XavierUniform([nO, nR],sd=40)
        self.BS = XavierUniform([nR, nO],sd=39)

        self.rls = RLS_LogarithmicRegressor(0.9,1e7)
        
        self.yP, self.yR, self.yL, self.yU = [np.array([]) for i in range(4)]
        self.eS = np.zeros(nI)
        self.eP = np.array([])
        self.eR = np.array([])

        self.εY, self.εM, self.εE, self.εR, self.ΣW = [0 for i in range(5)]
        self.εM_hist = np.array([])

        self.μrWAPE = 0
        self.MPsum = 0

        self.rR = 1e-9
        self.rP = 1e-10
        self.rL = 1e-11
        self.rU = 1e-12
        self.rRsum = 0

        self.rulR, self.rulP, self.rulL, self.rulU = [np.array([]) for i in range(4)]


    def PredSingle(self,x):

        u = np.dot(self.wR, self.hS) + np.dot(self.wI, x)
        h = self.hS + (-self.hS + Activation(u,self.act))/self.τ
        y = np.dot(self.wO, h)

        return y

    def fit(self,xP,yR,store=False,show=False):

        exp=2
        W = self.j**exp
        η1,η2,η3 = self.ηS        

        uS = self.wR @ self.hP + self.wI @ xP
        hP = self.hP + (-self.hP + Activation(uS,self.act))/self.τ
        yP = self.wO @ hP
        eS = yR-yP

        if show: 
            print('yR:',yR)
            print('yP:',yP)
        self.pS = np.outer(dActivation(uS,self.act),self.hP)/self.τ + (1-1/self.τ)*self.pS
        self.qS = np.outer(dActivation(uS,self.act),self.xPi)/self.τ + (1-1/self.τ)*self.qS

        δOS = η1*np.outer(eS,hP)
        δRS = η2*np.outer((self.BS@eS),np.ones(self.nR))*self.pS
        δIS = η3*np.outer(np.dot(self.BS, eS),np.ones(self.nI))*self.qS

        self.ΔIS = (self.ΔIS*(self.k-1) + δIS)/self.k
        self.ΔRS = (self.ΔRS*(self.k-1) + δRS)/self.k
        self.ΔOS = (self.ΔOS*(self.k-1) + δOS)/self.k

        self.wI = self.wI + δIS
        self.wR = self.wR + δRS
        self.wO = self.wO + δOS

        self.hP = hP
        self.xPi = xP

        self.UpdateRLS(yP,yR)

        ΣW = self.ΣW + W
        ΔY = np.abs((yR-yP)/yR)
        ΔM = np.linalg.norm(ΔY,ord=2)
        ΔR = np.abs((self.rR-self.rP)/(self.rR+1e-9))
        ΔE = np.abs((self.eR[-1]-self.eP[-1])/self.eR[-1])

        self.εY = ((self.εY*self.ΣW) + (W*ΔY[0]))/ΣW
        self.εM = ((self.εM*self.ΣW) + (W*ΔM))/ΣW
        self.εR = ((self.εR*self.ΣW) + (W*ΔR))/ΣW
        self.εE = ((self.εE*self.ΣW) + (W*ΔE))/ΣW
        self.ΣW = ΣW

        if store:
            self.yR = np.append(self.yR,yR[0])
            self.εM_hist = np.append(self.εM_hist,self.εM)

        self.k = self.k+1
        self.j = self.j+1
        self.t = np.append(self.t,self.k + self.nI)
        #self.ηS = self.ηS/(1 + self.decay*self.k)


    def PredRulIntr(self, x,lim=0.2,maxRul=110,store=False,show=False):
        xL,xP,xU =x.copy(), (x-(self.ρ*self.eS)).copy(),(x+(self.ρ*self.eS)).copy()        

        #print(xL[-5:])
        #print(xP[-5:])
        #print(xU[-5:])
        predict = True
        k=1
        PredRuls = [True for i in range(3)]
        PredVals, Ruls = [0 for i in range(3)], [0 for i in range(3)]
        #print(self.ht)
        wR,wI,wO = self.wR,self.wI,self.wO

        wRp, wRn = np.maximum(0, self.wR), np.abs(np.minimum(0, self.wR))
        wIp, wIn = np.maximum(0, self.wI), np.abs(np.minimum(0, self.wI))
        wOp, wOn = np.maximum(0, self.wO), np.abs(np.minimum(0, self.wO))
        #hU,hL = np.maximum(0, self.hS), np.minimum(0, self.hS)
        hL,hP,hU = [self.hP.copy() for i in range(3)]

        while predict:
            #print('hL:',hL[-5:])
            #print('hU:',hU[-5:])

            #print('xL:',xL[-3:],'xU:',xU[-3:])
            #print('xL:',xL[-3:],'xU:',xU[-3:])
            
            uP = wR@hP + wI@xP
            uL = (wRp @ hL - wRn @ hU) + (wIp @ xL - wIn @ xU)
            uU = (wRp @ hU - wRn @ hL) + (wIp @ xU - wIn @ xL)

            hP = hP*(1-1/self.τ) + Activation(uP,self.act)/self.τ
            hL = hL*(1-1/self.τ) + Activation(uL,self.act)/self.τ
            hU = hU*(1-1/self.τ) + Activation(uU,self.act)/self.τ
            
            yP = (wO@hP)
            yL = (wOp @ hL - wOn @ hU)
            yU = (wOp @ hU - wOn @ hL)
            #if show: print(yP)
            yP = yP[0]
            yL = yL[0]
            yU = yU[0]
        
            #print('hL:',hL[-5:])
            #print('hP:',hP[-5:])
            #print('hU:',hU[-5:])

            #print('xP:',xP[-3:],'yP:',yP[-4:])
            #print('xU:',xU[-3:],'yU:',yU)

            xP = np.delete(np.append(xP,yP),0)
            xL = np.delete(np.append(xL,yL),0)
            xU = np.delete(np.append(xU,yU),0)

            PredVals = [yL,yP,yU]

            #if show:
            #    print(PredVals)
            

            if k == 1:
                self.yL = np.append(self.yL,yL)
                self.yP = np.append(self.yP,yP)
                self.yU = np.append(self.yU,yU)
                #print(PredVals)
            k = k+1
            
            CheckPred,CheckLim=0,0

            for i in range(3):
                if PredRuls[i]: 
                    Ruls[i] = Ruls[i]+1
                    if Ruls[i] >= maxRul:
                        CheckLim = CheckLim + 1
                        Ruls[i] = maxRul
                        PredRuls[i] = False
                if PredVals[i] < lim: PredRuls[i] = False
                if not PredRuls[i]: CheckPred = CheckPred + 1
            if CheckPred == 3:break
            if CheckLim == 3:break
        
        self.rR=self.ref-self.k
        self.rL,self.rP,self.rU = Ruls

        if store:
            self.rulR = np.append(self.rulR,self.rR)
            self.rulL = np.append(self.rulL,self.rL)
            self.rulP = np.append(self.rulP,self.rP)
            self.rulU = np.append(self.rulU,self.rU)
    
    def PredRul(self, x,lim=0.2,store=False):
        xP = x.copy()
        rulP=0
        predict = True

        while predict:
            u = (self.wR @ self.hP) + (self.wI @ xP)
            hP = self.hP + (1/self.τ) * (-self.hP + Activation(u,self.act))
            yP = (self.wO @ hP)[-1]
            #print(yP)
            xP = np.delete(np.append(xP,yP),0)

            if predict: rulP = rulP+1
            if yP < lim: predict = False
            if rulP >= 110:
                rulP = 0
                break

        self.rR=self.ref-self.k
        self.rP = rulP

        if store:
            self.rulR = np.append(self.rulR,self.rR)
            self.rulP = np.append(self.rulP,self.rP)
    
    def UpdateRLS(self,yP,yR):
        eP = np.abs(self.rls.predict(np.abs(yP[-1])))
        eR = np.abs(yP-yR)[-1]
        self.rls.update(np.abs(yP[-1]), eR)
        self.eR = np.append(self.eR,eR)
        self.eP = np.append(self.eP,eP)
        self.eS = np.append(self.eS,eP)
        self.eS = np.delete(self.eS,0)


#Optimize parameters for minimize error of degradation prediction

In [46]:
rates = [1/(10**i) for i in range(1,7)][::-1]
def objective(trial):

    nI = trial.suggest_int('nI', 10, 19) 
    nR = trial.suggest_int('nR', 30, 43) 
    nO = trial.suggest_int('nO', 1, 7) 
    N1 = trial.suggest_categorical('N1', rates) 
    N2 = trial.suggest_categorical('N2', rates) 
    N3 = trial.suggest_categorical('N3', rates) 
    τ = trial.suggest_int('τ', 1, 12)        
    X,Y = PrepareDataAhead(sig,n=nI,m=nO)
    rnn = RTLO(nI,nR,nO,[N1,N2,N3],τ)
    rnn.ref = len(sig)-nI

    for i,_ in enumerate(X):
        rnn.fit(X[i],Y[i],store=True)

        if i == 50:
            if np.mean(rnn.εM_hist)>1:
                raise TrialPruned()

        if i % 20 == 0:  # Report every 10 time steps
            trial.report(rnn.εM, step=i)
            if trial.should_prune():
                raise optuna.TrialPruned()
            
    #return rnn.εY
    return rnn.εM

#pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10)
pruner=optuna.pruners.HyperbandPruner()

study = optuna.create_study(
    direction="minimize",
    sampler=SelSampler(mode='random'),
    pruner=pruner,
    #storage="sqlite:///" + f'Optuna/{FileName}_Prdct.db', study_name=f'P{4}',
    load_if_exists=True)
study.optimize(objective, n_trials=10000)
params = list(study.best_params.values())
print('Erro:', study.best_value, 'parameters: ', params)

[I 2026-04-15 13:00:50,526] A new study created in memory with name: no-name-f7f3e8d2-59bb-4f2d-831a-305cdc0fd036
[I 2026-04-15 13:00:50,538] Trial 0 finished with value: 0.055788192283043714 and parameters: {'nI': 11, 'nR': 36, 'nO': 7, 'N1': 0.1, 'N2': 0.001, 'N3': 0.1, 'τ': 9}. Best is trial 0 with value: 0.055788192283043714.
[I 2026-04-15 13:00:50,546] Trial 1 pruned. 
[I 2026-04-15 13:00:50,558] Trial 2 finished with value: 0.0549847964336794 and parameters: {'nI': 19, 'nR': 30, 'nO': 4, 'N1': 0.1, 'N2': 1e-05, 'N3': 0.001, 'τ': 9}. Best is trial 2 with value: 0.0549847964336794.
[I 2026-04-15 13:00:50,565] Trial 3 pruned. 
[I 2026-04-15 13:00:50,565] Trial 4 pruned. 
[I 2026-04-15 13:00:50,572] Trial 5 pruned. 
[I 2026-04-15 13:00:50,576] Trial 6 pruned. 
[I 2026-04-15 13:00:50,578] Trial 7 pruned. 
[I 2026-04-15 13:00:50,587] Trial 8 pruned. 
[I 2026-04-15 13:00:50,594] Trial 9 pruned. 
[I 2026-04-15 13:00:50,598] Trial 10 pruned. 
[I 2026-04-15 13:00:50,611] Trial 11 finished 

Erro: 0.00635226997024976 parameters:  [16, 34, 1, 0.1, 0.001, 1e-05, 2]


Erro: 0.010529050206394303 parameters:  [16, 34, 2, 0.1, 0.1, 0.0001, 2]\
Erro: 0.009681499917996990 parameters:  [16, 34, 2, 0.1, 0.01, 0.01, 3]\
Erro: 0.009530027489406095 parameters:  [16, 34, 2, 0.1, 1e-06,  0.001, 2]\
Erro: 0.00635226997024976 parameters:  [16, 34, 1, 0.1, 0.001, 1e-05, 2]\
Erro: 0.00635226997024976 parameters:  [16, 34, 1, 0.1, 0.001, 1e-05, 2]




In [3]:
params = [10, 34, 2, 0.01, 1e-07, 1e-05, 3]
params = {'nI': 29, 'nR': 14, 'nO': 24, 'N1': 1e-06, 'N2': 0.001, 'N3': 0.01, 'τ': 32}
params = list(params.values())
params = [22, 35, 2, 0.01, 1e-07, 0.01, 1]


In [49]:
nI,nR,nO,N1,N2,N3,τ= params
X,Y = PrepareDataAhead(sig,n=nI,m=nO)
rnn = RTLO(nI,nR,nO,[N1,N2,N3],τ)
rnn.ref = len(sig)-nI
for i in range(len(X[:])):
    #rnn.PredRul(x=X[i],store=True)
    rnn.PredRulIntr(x=X[i],store=True)
    rnn.fit(X[i],Y[i],store=True)
#print(.wR[0])

print(rnn.εY)  
print(rnn.εM)      
PlotPredErrorPLY(rnn)


0.00635226997024976
0.00635226997024976


In [29]:
i=50
err = np.mean(rnn.εM_hist[:i])

print('erro:',err)

erro: 0.6394576574566444


In [99]:
i=75
r_m = np.mean(rnn.rulR[:i])
r_mL = np.mean(rnn.rulR[:i])*0.3
r_mU = np.mean(rnn.rulR[:i])*1.15
p_m = (np.mean(rnn.rulP[:i]))

print('lower:',r_mL,'mid:',r_m,'upper:',r_mU)
print('pred:',p_m)

i=50
f=90
r_m = np.mean(rnn.rulR[i:f])
r_mL = np.mean(rnn.rulR[i:f])*0.3
r_mU = np.mean(rnn.rulR[i:f])*1.5
p_m = (np.mean(rnn.rulP[i:f]))

print('lower:',r_mL,'mid:',r_m,'upper:',r_mU)
print('pred:',p_m)

lower: 24.3 mid: 81.0 upper: 93.14999999999999
pred: 109.34666666666666
lower: 14.549999999999999 mid: 48.5 upper: 72.75
pred: 82.9


In [141]:
#params =  [14, 8, 14, 0.001, 0.01, 1e-06, 1e-07, 29]
nI,nR,nO,N1,N2,N3,τ= params
ηS = [N1,N2,N3]
X,Y = prepare_data(sig,n=nI,m=nO)
rnn = RTLO(nI,nR,nO,ηS,τ)
rnn.ref = len(sig)-nI

i=0

In [157]:
#rnn.PredRul(x=X[i],store=True)
rnn.PredRulIntr(x=X[i],store=True,show=False)

rnn.fit(X[i],Y[i],start=0,store=True,show=True)
i=i+1

yR: [0.83826309 0.83657452]
yP: [0.85561864 0.84293696]
